In [0]:
import logging
import re
from typing import List, Dict, Optional
from urllib.parse import urljoin

import dataiku
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as date_parser

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
TARGET_URL = "https://cdn.downloads.dataiku.com/public/dss-plugins/"
# Updated to match the dataset name seen in your logs
OUTPUT_DATASET_NAME = "store_plugin_ids_and_dss_versions" 
OUTPUT_CONNECTION = "postgres_localhost"
USER_AGENT = "DataikuDSS/14.2 (Scraping Recipe)"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# -----------------------------------------------------------------------------
# Core Scraping Logic (Unchanged)
# -----------------------------------------------------------------------------

class PluginDirectoryScraper:
    """
    Encapsulates logic to scrape the Dataiku CDN open directory for plugins.
    """
    def __init__(self, base_url: str):
        self.base_url = base_url
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": USER_AGENT})

    def _get_soup(self, url: str) -> Optional[BeautifulSoup]:
        """Fetches a URL and returns a BeautifulSoup object."""
        try:
            response = self.session.get(url, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.text, "html.parser")
        except requests.RequestException as e:
            logger.error(f"Failed to fetch {url}: {e}")
            return None

    def _is_valid_link(self, href: str, text: str) -> bool:
        """Filters out parent directory links and query parameters."""
        if not href or href.startswith("?") or href.startswith("/") or "Parent Directory" in text:
            return False
        return True

    def _parse_directory_row(self, row_text: str) -> Optional[str]:
        """Attempts to extract a date string from the text row of a directory listing."""
        date_match = re.search(r'(\d{4}-\d{2}-\d{2}|\d{2}-[a-zA-Z]{3}-\d{4})', row_text)
        if date_match:
            return row_text[date_match.start():].strip().split('  ')[0]
        return None

    def get_plugin_list(self) -> List[str]:
        """Scrapes the root URL for a list of plugin IDs."""
        soup = self._get_soup(self.base_url)
        if not soup:
            return []

        plugins = []
        for link in soup.find_all("a"):
            href = link.get("href")
            text = link.text
            if self._is_valid_link(href, text) and href.endswith("/"):
                plugins.append(href.strip("/"))
        return plugins

    def get_versions_for_plugin(self, plugin_id: str) -> List[Dict]:
        """Visits a specific plugin page and extracts version and date info."""
        plugin_url = urljoin(self.base_url, f"{plugin_id}/")
        soup = self._get_soup(plugin_url)
        if not soup:
            return []

        versions = []
        for link in soup.find_all("a"):
            href = link.get("href")
            text = link.text

            if not self._is_valid_link(href, text):
                continue

            version_number = href.strip("/")
            date_str = None
            
            # Strategy 1: Next sibling text
            if link.next_sibling and isinstance(link.next_sibling, str):
                raw_text = link.next_sibling.strip()
                date_str = self._parse_directory_row(raw_text)
            
            # Strategy 2: Table cells
            if not date_str:
                parent_td = link.find_parent("td")
                if parent_td:
                    next_td = parent_td.find_next_sibling("td")
                    if next_td:
                        date_str = next_td.text.strip()

            parsed_date = None
            if date_str:
                try:
                    parsed_date = date_parser.parse(date_str)
                except Exception:
                    pass

            versions.append({
                "plugin-id": plugin_id,
                "dss-version-number": version_number,
                "last_modified": parsed_date
            })

        return versions

    def run(self) -> pd.DataFrame:
        """Main execution method."""
        logger.info(f"Starting scrape of {self.base_url}")
        all_data = []
        plugin_ids = self.get_plugin_list()
        logger.info(f"Found {len(plugin_ids)} plugins.")

        for pid in plugin_ids:
            versions = self.get_versions_for_plugin(pid)
            all_data.extend(versions)

        return pd.DataFrame(all_data)

# -----------------------------------------------------------------------------
# Dataiku Integration (Updated for Managed PostgreSQL)
# -----------------------------------------------------------------------------

def ensure_managed_dataset_exists(project, dataset_name: str, connection_name: str):
    """
    Checks if a dataset exists. If not, creates it as a MANAGED dataset 
    on the specified SQL connection.
    
    Using create_managed_dataset is crucial for SQL targets to allow 
    overwriting/dropping tables without 'External' dataset permission errors.
    """
    existing_datasets = [d['name'] for d in project.list_datasets()]
    
    if dataset_name in existing_datasets:
        logger.info(f"Dataset '{dataset_name}' already exists.")
        # Optional: Check if the existing dataset actually points to the right connection
        # but for this recipe, we assume the name ownership is sufficient.
    else:
        logger.info(f"Dataset '{dataset_name}' does not exist. Creating managed dataset on {connection_name}...")
        try:
            project.create_managed_dataset(dataset_name, connection_name)
            logger.info(f"Dataset '{dataset_name}' successfully created.")
        except Exception as e:
            logger.error(f"Failed to create managed dataset: {e}")
            raise

def save_to_dss(df: pd.DataFrame, dataset_name: str, connection_name: str):
    """
    Orchestrates the saving of data to DSS.
    """
    client = dataiku.api_client()
    project = client.get_default_project()
    
    # 1. Ensure Managed Dataset Exists on PostgreSQL
    ensure_managed_dataset_exists(project, dataset_name, connection_name)
    
    # 2. Write Data
    logger.info(f"Writing {len(df)} rows to {dataset_name} on connection {connection_name}")
    dss_dataset = dataiku.Dataset(dataset_name)
    
    # write_with_schema will automatically handle the DROP/CREATE TABLE logic 
    # because the dataset is now Managed.
    dss_dataset.write_with_schema(df)
    logger.info("Write complete.")

# -----------------------------------------------------------------------------
# Main Entry Point
# -----------------------------------------------------------------------------

def main():
    scraper = PluginDirectoryScraper(TARGET_URL)
    df_plugins = scraper.run()
    
    if not df_plugins.empty:
        # Clean date column for DSS/PostgreSQL compatibility
        if "last_modified" in df_plugins.columns:
            # Coerce to datetime; errors='coerce' turns unparseable data to NaT (NULL in SQL)
            df_plugins["last_modified"] = pd.to_datetime(df_plugins["last_modified"], errors='coerce')
            
        save_to_dss(df_plugins, OUTPUT_DATASET_NAME, OUTPUT_CONNECTION)
    else:
        logger.warning("No data found during scrape. Dataset update skipped.")

if __name__ == "__main__":
    main()